In [23]:
import ast
import datetime
import json
import numpy as np
import pandas as pd
import re
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
import spacy
from spacy.tokens import Span, Doc
from spacy import displacy
from typing import List

from dap_job_quality.utils.keyword_search_patterns import keywords
from dap_job_quality.getters.ojo_getters import get_ojo_sample
from dap_job_quality.utils.spacy_keyword_search import get_matches, get_spans
from dap_job_quality.utils.text_cleaning import clean_text

from dap_job_quality.getters.data_getters import load_s3_jsonl
from dap_job_quality.getters.labelled_data import get_labelled_job_sentences
from dap_job_quality.utils import prodigy_data_utils as pdu

from dap_job_quality import BUCKET_NAME, PROJECT_DIR, config

model = SentenceTransformer("all-MiniLM-L6-v2")

pd.set_option('display.width', 1000)

TODAY = datetime.datetime.today().strftime('%Y-%m-%d')

nlp = spacy.load("en_core_web_sm")

SEED = config["seed"]

2024-06-11 08:11:22,544 - sentence_transformers.SentenceTransformer - INFO - Load pretrained SentenceTransformer: all-MiniLM-L6-v2
2024-06-11 08:11:22,718 - sentence_transformers.SentenceTransformer - INFO - Use pytorch device: cpu


In [3]:
lookup = pd.read_csv(PROJECT_DIR / "inputs/keyword_lookup - v5.csv")

In [4]:
# Find the max length of target_phrase in the lookup table.
# We would want to split sentences into ngrams this length
def count_words(text):
    return len(text.split())

lookup['word_count'] = lookup['target_phrase'].apply(count_words)

max_word_count = lookup['word_count'].max()

max_word_count

4

In [40]:
def generate_ngrams(text, n=4) -> List[str]:
    """Split a text into ngrams.

    Args:
        text (str): The text to be split
        n (int, optional): The length of the ngrams. Defaults to 4.

    Returns:
        List[str]: A list of ngrams
    """
    words = text.split()
    ngrams = [' '.join(words[i:i+n]) for i in range(len(words) - n + 1)]
    return ngrams

def split_text(text):
    # Regular expression pattern to match ',', ';', ':', or '- ' as long as it is not part of a number
    pattern = r'(?<![\d£$€])[;,:!]|(?<![\d£$€]) - '
    # Use re.split() to split the text on the pattern
    return re.split(pattern, text)

def split_text_on_cc(text):
    """Split text on a coordinating conjunction (CC) token
    **if** the tokens immediately to the left and right of the 'CC' token have different POS tags.
    
    So we don't want to split "professional and personal development" but we would split "competitive salary and you will have the opportunity to work from home"
    """
    # Process the text with spaCy
    doc = nlp(text)
    
    # Find the indices of tokens with the 'CC' dependency label
    cc_indices = [token.i for token in doc if token.dep_ == 'cc']
    
    # Initialize the start index and list for split segments
    start_idx = 0
    segments = []
    
    # Split the text at each 'CC' token
    for idx in cc_indices:
        # Check if the tokens immediately to the left and right of 'CC' have the same POS
        if idx > 0 and idx < len(doc) - 1:
            left_token = doc[idx - 1]
            right_token = doc[idx + 1]
            if left_token.pos_ != right_token.pos_:
                segments.append(doc[start_idx:idx].text.strip())
                start_idx = idx + 1
    
    # Append the last segment
    segments.append(doc[start_idx:].text.strip())
    
    return segments

def split_text_on_phrases(text):
    # Regular expression pattern to match the phrases "and a" or "with a"
    pattern = r'with the|with a|and the|and a'
    # Use re.split() to split the text on the pattern
    return re.split(pattern, text)

In [31]:
sample_sents_alt = ['In return you will receive an attractive package, long term work opportunities and a clear path to progress to Project manager with a 6 month performance review.',
                'This role will be based in one of our offices (London, Cardiff, Edinburgh).',
                'We follow a hybrid working arrangement with a minimum of two days in the office.',
                'Join us to develop your strengths and enjoy a fulfilling career full of varied experiences',
                'The organisation also gets the global team together once in the summer and once for the Christmas party - usually an international trip to say thank you.',
                'The role is 36 hours per week, working 5 days out of 7.',
                'Free onsite parking and access to electric charging points.',
                'My client is offering £25,000 - £30,000 plus benefits depending on experience with the ability to work from home 2 3 days a week.',
                'They offer an ego and politics free working environment and subsequently enjoy a high retention rate',
                'Professional and personal learning and development opportunities.',
                'The opportunity to work for a leading international corporation',
                'FURTHER INFORMATION One year fixed-term contracts) working at The St Marylebone CE Bridge School (West London).',
                'Opportunity to earn monthly commission Avis Budget Group is a leading global provider of mobility solutions, operating three of the most recognized brands in the industry through Avis, Budget and Zipcar, the world’s leading car-sharing network.',
                "- In 2012 the school was named amongst the top 100 non-selective schools -87% A. - C at GCSE including English and Math's -Very high level of pupil behaviour -Exceptional facilities and resources -Paid in line with MPS UPS REQUIREMENTS Applications are welcome from teachers at any stage in their career including NQT's and both British trained Spanish teachers and overseas Spanish teachers will be considered for this position."]

In [21]:
split_text_on_cc("professional and personal learning and development opportunities, and a warm and friendly environment.")

['professional and personal learning and development opportunities,',
 'a warm and friendly environment.']

In [55]:
def chunk_text(text, ngrams=True):
    phrases = []
    
    sent = split_text_on_cc(text)
    for sub_sent in sent:
        sent = split_text(sub_sent)
        for sub_sub_sent in sent:
            phrases.extend(split_text_on_phrases(sub_sub_sent))
    
    if ngrams:
        for phrase in phrases:
            if count_words(phrase) > 6:
                ngrams = generate_ngrams(phrase, n=6)
                phrases.extend(ngrams)
            
    return phrases

chunk_text('My client is offering £25,000 - £30,000 plus benefits depending on experience with the ability to work from home 2 3 days a week. They offer an ego and politics free working environment and subsequently enjoy a high retention rate. Professional and personal learning and development opportunities. The opportunity to work for a leading international corporation')

['My client is offering £25,000 - £30,000',
 'benefits depending on experience ',
 ' ability to work from home 2 3 days a week. They offer an ego and politics free working environment',
 'subsequently enjoy a high retention rate. Professional and personal learning and development opportunities. The opportunity to work for a leading international corporation',
 'My client is offering £25,000 -',
 'client is offering £25,000 - £30,000',
 'ability to work from home 2',
 'to work from home 2 3',
 'work from home 2 3 days',
 'from home 2 3 days a',
 'home 2 3 days a week.',
 '2 3 days a week. They',
 '3 days a week. They offer',
 'days a week. They offer an',
 'a week. They offer an ego',
 'week. They offer an ego and',
 'They offer an ego and politics',
 'offer an ego and politics free',
 'an ego and politics free working',
 'ego and politics free working environment',
 'subsequently enjoy a high retention rate.',
 'enjoy a high retention rate. Professional',
 'a high retention rate. Profe

In [59]:
# This is how we could apply the function to a column of sentences in a dataframe?
sentence_chunks = []

for sent in sample_sents_alt:
    chunks = chunk_text(sent, ngrams=True)
    sentence_chunks.append(chunks)
    
chunked_df = pd.DataFrame(zip(sample_sents_alt, sentence_chunks))

In [60]:
chunked_df.iloc[0, 0]

'In return you will receive an attractive package, long term work opportunities and a clear path to progress to Project manager with a 6 month performance review.'

In [61]:
chunked_df.iloc[0, 1]

['In return you will receive an attractive package',
 ' long term work opportunities',
 'a clear path to progress to Project manager ',
 ' 6 month performance review.',
 'In return you will receive an',
 'return you will receive an attractive',
 'you will receive an attractive package',
 'a clear path to progress to',
 'clear path to progress to Project',
 'path to progress to Project manager']